# 01_0 Repair Baseline Comparison

This notebook runs the control conditions needed to separate source repair effects from pseudoGT and DQA effects.

Conditions:
- `repair_only`: warmup -> source repair repeated for the same number of rounds.
- `pseudo_fedavg`: pseudoGT clients -> plain FedAvg -> source repair.
- `pseudo_dqa`: pseudoGT clients -> DQA-CWA v2 -> source repair.

The runner sends Discord notifications at start and finish when `--notify` is enabled.

In [1]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "dynamic_quality_aware_classwise_aggregation").exists():
    PROJECT_ROOT = cwd / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa"
else:
    PROJECT_ROOT = cwd

REPO_ROOT = PROJECT_ROOT.parents[1]
WORKSPACE = PROJECT_ROOT / "output" / "01_0_repair_baseline_comparison"
print("PROJECT_ROOT", PROJECT_ROOT)
print("WORKSPACE", WORKSPACE)

PROJECT_ROOT /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa
WORKSPACE /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison


## Setup Only

Run this first if you only want to build the six client/eval lists and confirm paths.

In [2]:
CLIENT_LIMIT = 1500
subprocess.run([
    sys.executable,
    "scripts/run_scene_daynight_dqa_01_0.py",
    "--workspace-root", str(WORKSPACE),
    "--client-limit", str(CLIENT_LIMIT),
    "--conditions", "all",
    "--setup-only",
], cwd=PROJECT_ROOT, check=True)



######## condition=repair_only ########
Workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/repair_only
Warmup: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/repair_only/global_checkpoints/round000_warmup.pt
{
  "clients": [
    {
      "id": 0,
      "name": "highway_day",
      "weather": "highway_day",
      "scene": "highway",
      "timeofday": "daytime"
    },
    {
      "id": 1,
      "name": "highway_night",
      "weather": "highway_night",
      "scene": "highway",
      "timeofday": "night"
    },
    {
      "id": 2,
      "name": "citystreet_day",
      "weather": "citystreet_day",
      "scene": "city street",
      "timeofday": "daytime"
    },
    {
      "id": 3,
      "name": "citystreet_night",
      "weather": "citystreet_night",
      "scene": "city street",
      "timeofday": "night"
    },
    {
      

CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', 'scripts/run_scene_daynight_dqa_01_0.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison', '--client-limit', '1500', '--conditions', 'all', '--setup-only'], returncode=0)

## Run Baselines

Default is 3 rounds for all three conditions.  This can be long because `pseudo_fedavg` and `pseudo_dqa` each train six clients per round.

In [3]:
ROUNDS = 3
CONDITIONS = "all"  # repair_only,pseudo_fedavg,pseudo_dqa
MAX_IMAGES_PER_CLIENT = 0
EVAL_CLIENTS = False

cmd = [
    sys.executable,
    "scripts/run_scene_daynight_dqa_01_0.py",
    "--workspace-root", str(WORKSPACE),
    "--client-limit", str(CLIENT_LIMIT),
    "--conditions", CONDITIONS,
    "--rounds", str(ROUNDS),
    "--batch-size", "160",
    "--workers", "8",
    "--gpus", "2",
    "--master-port", "30741",
    "--server-repair-epochs", "1",
    "--evaluate",
    "--classwise",
    "--no-eval-plots",
    "--notify",
]
if MAX_IMAGES_PER_CLIENT > 0:
    cmd.extend(["--max-images-per-client", str(MAX_IMAGES_PER_CLIENT)])
if EVAL_CLIENTS:
    cmd.append("--eval-clients")

print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

/root/micromamba/envs/al_yolov8/bin/python scripts/run_scene_daynight_dqa_01_0.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison --client-limit 1500 --conditions all --rounds 3 --batch-size 160 --workers 8 --gpus 2 --master-port 30741 --server-repair-epochs 1 --evaluate --classwise --no-eval-plots --notify
DiscordNotifyResult(ok=True, chunks_sent=1, status_codes=(204,), dry_run=False, error=None)


######## condition=repair_only ########
Reusing completed condition metrics: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/repair_only/stats/01_0_condition_metrics.csv


######## condition=pseudo_fedavg ########
Workspace: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/pseudo_fedavg
Warmup: /app/Object_Detection/dynamic_quality_aware_classwise_agg

EfficientTeacher  e8b42eb torch 2.8.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48508.9375MB)

Fusing layers... 
Model summary: 365 layers, 46156743 parameters, 0 gradients
/root/micromamba/envs/al_yolov8/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


round001 client0_highway_day: pseudo scan 1/1500 images, kept 12 boxes
round001 client0_highway_day: pseudo scan 250/1500 images, kept 2318 boxes
round001 client0_highway_day: pseudo scan 500/1500 images, kept 4696 boxes
round001 client0_highway_day: pseudo scan 750/1500 images, kept 7005 boxes
round001 client0_highway_day: pseudo scan 1000/1500 images, kept 9304 boxes
round001 client0_highway_day: pseudo scan 1250/1500 images, kept 11580 boxes
round001 client0_highway_day: pseudo scan 1500/1500 images, kept 13881 boxes
{
  "round": "round001",
  "client": "client0_highway_day",
  "teacher": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/pseudo_fedavg/global_checkpoints/round000_warmup.pt",
  "source_images_scanned": 1500,
  "pseudo_images_kept": 1488,
  "pseudo_boxes_kept": 10812,
  "boxes_per_kept_image": 7.266129032258065,
  "mean_conf": 0.7457716877548465,
  "mean_stability": 0.9449824729243165,
  "mean_s

EfficientTeacher  e8b42eb torch 2.8.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48508.9375MB)

Fusing layers... 
Model summary: 365 layers, 46156743 parameters, 0 gradients


round002 client0_highway_day: pseudo scan 1/1500 images, kept 12 boxes
round002 client0_highway_day: pseudo scan 250/1500 images, kept 2355 boxes
round002 client0_highway_day: pseudo scan 500/1500 images, kept 4781 boxes
round002 client0_highway_day: pseudo scan 750/1500 images, kept 7121 boxes
round002 client0_highway_day: pseudo scan 1000/1500 images, kept 9486 boxes
round002 client0_highway_day: pseudo scan 1250/1500 images, kept 11808 boxes
round002 client0_highway_day: pseudo scan 1500/1500 images, kept 14171 boxes
{
  "round": "round002",
  "client": "client0_highway_day",
  "teacher": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/pseudo_fedavg/checkpoints/round001_repair_oriented_all_lowlr_server_repair.pt",
  "source_images_scanned": 1500,
  "pseudo_images_kept": 1494,
  "pseudo_boxes_kept": 11016,
  "boxes_per_kept_image": 7.373493975903615,
  "mean_conf": 0.781026014937771,
  "mean_stability": 0.95

...


Validating /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/pseudo_dqa/runs/pl03_round003_repair_oriented_all_lowlr_server_repair/weights/best.pt...
Fusing layers... 
Model summary: 365 layers, 46156743 parameters, 0 gradients
               Class     Images     Labels          P          R     mAP@.5 mAP@.5:.95: 100%|██████████| 5/5 [00:04<00:00,  1.08it/s]


                 all        738      14937      0.442      0.495      0.409       0.23
                   0        738       1052      0.405       0.59      0.541      0.258
                   1        738         44      0.248      0.295      0.155      0.061
                   2        738       8622      0.543      0.771      0.753      0.487
                   3        738        151      0.279      0.464      0.418      0.317
                   4        738        467      0.406      0.621      0.556      0.404
                   5        738         70      0.244      0.514      0.287      0.138
                   6        738         65      0.298      0.385      0.176     0.0632
                   7        738       1619      0.488       0.65       0.59      0.243
                   8        738       2845      0.507      0.665      0.617      0.327
                   9        738          2          1          0   0.000325   0.000293


Results saved to /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/pseudo_dqa/runs/pl03_round003_repair_oriented_all_lowlr_server_repair

1 epochs completed in 0.028 hours.
Destroying process group... 
[rank0]:[W505 12:52:55.696364158 ProcessGroupNCCL.cpp:1538] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


/root/micromamba/envs/al_yolov8/bin/python /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/scripts/evaluate_scene_daynight_protocol.py --workspace /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/pseudo_dqa --splits highway_day,highway_night,citystreet_day,citystreet_night,residential_day,residential_night,total --batch-size 16 --no-plots --verbose --checkpoint warmup_global=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/pseudo_dqa/global_checkpoints/round000_warmup.pt --checkpoint round001_dqa_aggregate=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison/pseudo_dqa/checkpoints/round001_dqa_cwa_v2_aggregate.pt --checkpoint round001_server_repair=/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/

CompletedProcess(args=['/root/micromamba/envs/al_yolov8/bin/python', 'scripts/run_scene_daynight_dqa_01_0.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/output/01_0_repair_baseline_comparison', '--client-limit', '1500', '--conditions', 'all', '--rounds', '3', '--batch-size', '160', '--workers', '8', '--gpus', '2', '--master-port', '30741', '--server-repair-epochs', '1', '--evaluate', '--classwise', '--no-eval-plots', '--notify'], returncode=0)

## Combined Metrics

In [4]:
import pandas as pd

metrics_csv = WORKSPACE / "stats" / "01_0_all_condition_metrics.csv"
if metrics_csv.exists():
    df = pd.read_csv(metrics_csv)
    display(df)
else:
    print("No combined metrics yet:", metrics_csv)

,condition,round,aggregate_map50,aggregate_map50_95,repaired_map50,repaired_map50_95,repair_gain_map50_95,retained_gain_map50_95,round_delta_map50_95,worst_split,worst_split_map50_95,day_avg_map50_95,night_avg_map50_95,day_night_gap_map50_95
0,repair_only,1,NaN,NaN,0.373,0.206,NaN,0.018,NaN,highway_night,0.148,0.226333,0.167667,0.058667
1,repair_only,2,NaN,NaN,0.379,0.210,NaN,0.022,0.004,highway_night,0.151,0.230000,0.172333,0.057667
2,repair_only,3,NaN,NaN,0.381,0.212,NaN,0.024,0.002,highway_night,0.152,0.232333,0.173333,0.059000
3,pseudo_fedavg,1,0.350,0.196,0.377,0.209,0.013,0.021,NaN,highway_night,0.148,0.230000,0.167667,0.062333
4,pseudo_fedavg,2,0.349,0.198,0.379,0.211,0.013,0.023,0.002,highway_night,0.146,0.234000,0.167333,0.066667
5,pseudo_fedavg,3,0.345,0.196,0.379,0.212,0.016,0.024,0.001,highway_night,0.143,0.235667,0.163667,0.072000
6,pseudo_dqa,1,0.351,0.189,0.373,0.206,0.017,0.018,NaN,highway_night,0.148,0.226333,0.167333,0.059000
7,pseudo_dqa,2,0.373,0.206,0.379,0.210,0.004,0.022,0.004,highway_night,0.151,0.230000,0.172333,0.057667
8,pseudo_dqa,3,0.378,0.210,0.381,0.211,0.001,0.023,0.001,highway_night,0.151,0.232667,0.172667,0.060000


## Final-Round Comparison

In [5]:
if metrics_csv.exists():
    df = pd.read_csv(metrics_csv)
    final = df.sort_values("round").groupby("condition").tail(1)
    display(final[[
        "condition", "round", "aggregate_map50", "aggregate_map50_95",
        "repaired_map50", "repaired_map50_95", "retained_gain_map50_95",
        "worst_split", "worst_split_map50_95", "day_night_gap_map50_95",
    ]].sort_values("repaired_map50_95", ascending=False))

,condition,round,aggregate_map50,aggregate_map50_95,repaired_map50,repaired_map50_95,retained_gain_map50_95,worst_split,worst_split_map50_95,day_night_gap_map50_95
2,repair_only,3,NaN,NaN,0.381,0.212,0.024,highway_night,0.152,0.059
5,pseudo_fedavg,3,0.345,0.196,0.379,0.212,0.024,highway_night,0.143,0.072
8,pseudo_dqa,3,0.378,0.210,0.381,0.211,0.023,highway_night,0.151,0.060


## Split-Level Evaluation

In [6]:
rows = []
for condition in ["repair_only", "pseudo_fedavg", "pseudo_dqa"]:
    path = WORKSPACE / condition / "validation_reports" / "paper_protocol_eval_summary.csv"
    if not path.exists():
        continue
    part = pd.read_csv(path)
    part["condition"] = condition
    rows.append(part)
if rows:
    eval_df = pd.concat(rows, ignore_index=True)
    ok = eval_df[eval_df["status"].eq("ok")].copy()
    final_rows = ok[ok["checkpoint_label"].eq("round003_server_repair")]
    display(final_rows[["condition", "split", "precision", "recall", "map50", "map50_95"]].sort_values(["split", "condition"]))
else:
    print("No evaluation summaries yet.")

,condition,split,precision,recall,map50,map50_95
121,pseudo_dqa,citystreet_day,0.508,0.428,0.412,0.233
72,pseudo_fedavg,citystreet_day,0.517,0.430,0.416,0.236
23,repair_only,citystreet_day,0.510,0.426,0.412,0.233
122,pseudo_dqa,citystreet_night,0.468,0.353,0.327,0.174
73,pseudo_fedavg,citystreet_night,0.507,0.328,0.313,0.167
24,repair_only,citystreet_night,0.471,0.352,0.327,0.174
119,pseudo_dqa,highway_day,0.562,0.376,0.375,0.211
70,pseudo_fedavg,highway_day,0.588,0.373,0.379,0.215
21,repair_only,highway_day,0.563,0.375,0.376,0.211
120,pseudo_dqa,highway_night,0.479,0.265,0.276,0.151


## PseudoGT Stats

In [7]:
pseudo_rows = []
for condition in ["pseudo_fedavg", "pseudo_dqa"]:
    stats_dir = WORKSPACE / condition / "stats"
    for path in sorted(stats_dir.glob("03_round*_pseudo_label_stats.csv")):
        part = pd.read_csv(path)
        part["condition"] = condition
        pseudo_rows.append(part)
if pseudo_rows:
    pseudo_df = pd.concat(pseudo_rows, ignore_index=True)
    display(pseudo_df[[
        "condition", "round", "client", "pseudo_images_kept", "pseudo_boxes_kept",
        "boxes_per_kept_image", "mean_conf", "mean_stability", "mean_score",
    ]])
else:
    print("No pseudoGT stats yet.")

,condition,round,client,pseudo_images_kept,pseudo_boxes_kept,boxes_per_kept_image,mean_conf,mean_stability,mean_score
0,pseudo_fedavg,round001,client0_highway_day,1488,10812,7.2661,0.745772,0.944982,0.707788
1,pseudo_fedavg,round001,client1_highway_night,1476,9263,6.2757,0.645482,0.922016,0.598920
2,pseudo_fedavg,round001,client2_citystreet_day,1496,13789,9.2172,0.734744,0.940587,0.694337
3,pseudo_fedavg,round001,client3_citystreet_night,1484,12497,8.4212,0.646182,0.919555,0.598080
4,pseudo_fedavg,round001,client4_residential_day,1488,9829,6.6055,0.771392,0.951192,0.736932
5,pseudo_fedavg,round001,client5_residential_night,1486,9287,6.2497,0.702775,0.932827,0.658566
6,pseudo_fedavg,round002,client0_highway_day,1494,11016,7.3735,0.781026,0.954040,0.748264
7,pseudo_fedavg,round002,client1_highway_night,1462,8197,5.6067,0.679126,0.938476,0.641116
8,pseudo_fedavg,round002,client2_citystreet_day,1496,13620,9.1043,0.774709,0.951685,0.740573
9,pseudo_fedavg,round002,client3_citystreet_night,1472,11373,7.7262,0.677078,0.936137,0.637848
